# Classes, dataclasses and enums

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-05-classes-and-dataclasses.ipynb)

Up to now you have worked with values that Python gives you: numbers, strings, lists and dictionaries. A **class**
lets you define your own kind of value, with its own data and its own functions. Pipelines use classes for
connections, settings, records and pluggable steps. This chapter starts from the basics: what a class is, how to write
one, and then the shortcuts Python adds on top (dataclasses and enums).

## What is a class?

A **class** is a blueprint for a kind of object. An **object** (also called an **instance**) is one value built from that
blueprint. Each object has its own data, called **attributes**, and can run the functions defined in the class, called
**methods**.

A `dict` is a useful comparison. `{"name": "Ana"}` holds data, but it has no rules about what keys it must contain and
no behaviour attached. A class puts the data, the rules and the behaviour in one place.

You already use classes every day: a `list` is an instance of the class `list`, and `type([])` tells you so.

### Where you'll use it

- **Connectors and clients:** a database client class holds its host, credentials and retry settings in one object, and its methods run the queries.
- **Pipeline steps:** a `Transform` object holds its configuration and exposes one `run(batch)` method.

In [ ]:
numbers = [1, 2, 3]

assert type(numbers) is list          # numbers is an instance of the class list
assert isinstance(numbers, list)
assert numbers.append is not None     # append is a method that belongs to every list

## Your first class

The smallest class uses the `class` keyword, a name written in `CamelCase`, and a colon. The body is indented. The
`pass` statement means "nothing here yet".

Creating an object is called **instantiation**: you call the class like a function. The result is a new instance.

In [ ]:
class Customer:
    pass

ana = Customer()                 # create an instance
ben = Customer()                 # a second, separate instance

assert type(ana) is Customer
assert ana is not ben            # two different objects, even though they came from the same blueprint

## The constructor: `__init__` sets up the data

`__init__` is the **constructor**. Python calls it automatically right after an object is created, and it sets the
starting attributes. Its first parameter is always `self`, which means "this object". Inside the class you write
`self.name = name` to store the value on that object.

Each instance keeps its own copy of the attributes, so changing one customer does not change another.

In [ ]:
class Customer:
    def __init__(self, name, city):
        self.name = name         # store the value on this object
        self.city = city

ana = Customer("Ana Lopez", "Oslo")
ben = Customer("Ben Smith", "Lima")

assert ana.name == "Ana Lopez"   # read an attribute with a dot
assert ben.city == "Lima"

ana.city = "Madrid"              # change one object's attribute
assert ana.city == "Madrid"
assert ben.city == "Lima"        # the other object is untouched

## Methods: functions that belong to the object

A **method** is a function defined inside the class. It always receives `self` as its first argument, so it can read and
change the object's attributes. You call it with a dot: `ana.greeting()`. Python passes `ana` in as `self` for you.

### Where you'll use it

- **Domain behaviour next to data:** an `Invoice` object that knows how to compute its own total and tax, so the rule lives in one place.
- **Testability:** a method on a small object is tested by creating the object and calling the method, with no network or database.

In [ ]:
class Customer:
    def __init__(self, name, city):
        self.name = name
        self.city = city
        self.orders = []

    def greeting(self):
        return f"Hello, {self.name} from {self.city}"

    def add_order(self, amount):
        self.orders.append(amount)

    def total_spent(self):
        return sum(self.orders)

ana = Customer("Ana Lopez", "Oslo")
assert ana.greeting() == "Hello, Ana Lopez from Oslo"

ana.add_order(120)
ana.add_order(45.5)
assert ana.total_spent() == 165.5
assert Customer("Ben", "Lima").total_spent() == 0    # a new object starts with no orders

## Printing an object: `__repr__`

By default, printing an object shows only its type and memory address, which tells you nothing. The special method
`__repr__` returns the text shown in logs and in the notebook. Write it so that a log line says what the object holds.

Method names with two underscores on each side are called **dunder methods** (double underscore). Python calls them
for you in certain situations, such as `print` or `==`.

In [ ]:
class Customer:
    def __init__(self, name, city):
        self.name = name
        self.city = city

    def __repr__(self):
        return f"Customer(name={self.name!r}, city={self.city!r})"

ana = Customer("Ana Lopez", "Oslo")
assert repr(ana) == "Customer(name='Ana Lopez', city='Oslo')"
assert str(ana) == repr(ana)   # str() falls back to __repr__ when __str__ is not defined

## Equality: when are two objects equal?

By default, `==` on two objects is `True` only if they are the same object. For a record such as a customer, you often want
"same name and city means equal". Define `__eq__` to say so.

In [ ]:
class Customer:
    def __init__(self, name, city):
        self.name = name
        self.city = city

    def __eq__(self, other):
        if not isinstance(other, Customer):
            return NotImplemented
        return (self.name, self.city) == (other.name, other.city)

assert Customer("Ana", "Oslo") == Customer("Ana", "Oslo")   # equal values
assert Customer("Ana", "Oslo") != Customer("Ana", "Lima")
assert Customer("Ana", "Oslo") is not Customer("Ana", "Oslo")  # but two different objects

## Class attributes versus instance attributes

An instance attribute belongs to one object (`self.name`). A **class attribute** is written in the class body, outside
any method, and is shared by all instances. Read it through the class or through an instance.

Assigning `self.x` inside a method creates an instance attribute, even if a class attribute with the same name exists.

### Where you'll use it

- **Default settings:** a class-level default timeout that each instance can override.
- **Counters:** a class attribute counting how many objects were created, used for metrics in long-running workers.

In [ ]:
class Pipeline:
    default_retries = 3            # class attribute: one value for the whole class

    def __init__(self, name):
        self.name = name           # instance attribute: one value per object
        self.retries = Pipeline.default_retries

a, b = Pipeline("a"), Pipeline("b")
assert a.default_retries == 3       # readable through an instance too

Pipeline.default_retries = 5        # change the class attribute
assert a.retries == 3 and b.retries == 3    # already-created objects keep their copy
assert Pipeline("c").retries == 5           # new objects see the new value

## Properties: validate on assignment

A **property** looks like a plain attribute (`p.days`) but runs code when you read or write it. Use it to keep an object
valid: reject a bad value at the moment it is assigned, rather than failing much later.

- `@property` marks the getter (the code that runs on read).
- `@days.setter` marks the setter (the code that runs on `p.days = ...`).

### Where you'll use it

- **Guarding invariants:** a partition size, a retry count or a sample rate that must always be valid, rejected at the point it is set.

In [ ]:
class Partition:
    def __init__(self, days):
        self.days = days           # goes through the setter below

    @property
    def days(self):
        return self._days

    @days.setter
    def days(self, value):
        if value < 1:
            raise ValueError("a partition covers at least one day")
        self._days = value

p = Partition(7)
assert p.days == 7
try:
    p.days = 0
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

## Inheritance: reuse a class and change part of it

A class can **inherit** from another. The new class (the **subclass**) gets every attribute and method of the parent
(the **base class**), and can override some of them. Write the parent as the general idea and each child as one specific
case.

`super()` calls the parent's version of a method, so the child can add to it instead of copying it.

### Where you'll use it

- **Pluggable sources and sinks:** a `Source` base class with `CsvSource`, `S3Source` and `ApiSource`; the pipeline calls `read()` on any of them.
- **Framework hooks:** Airflow operators and many ORM base classes are designed around subclassing.

In [ ]:
class Source:
    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"source {self.name}"

class CsvSource(Source):
    def __init__(self, name, path):
        super().__init__(name)      # let the parent set self.name
        self.path = path

    def describe(self):
        return f"{super().describe()} reading {self.path}"

src = CsvSource("orders", "landing/orders.csv")
assert src.name == "orders"                       # inherited attribute
assert src.describe() == "source orders reading landing/orders.csv"
assert isinstance(src, Source)                    # a CsvSource is also a Source

## Polymorphism: one call, many classes

Because every subclass has the same method name, code that calls the method does not need to know which class it has.
This is **polymorphism**. A pipeline can loop over different sources and call `read()` on each one.

In [ ]:
class Source:
    def read(self):
        raise NotImplementedError("subclasses must implement read()")

class CsvSource(Source):
    def read(self):
        return ["csv row"]

class ApiSource(Source):
    def read(self):
        return ["api row"]

def load_all(sources):
    return [row for source in sources for row in source.read()]

assert load_all([CsvSource(), ApiSource()]) == ["csv row", "api row"]

try:
    Source().read()                 # the base class does not know how to read
except NotImplementedError:
    pass
else:
    raise AssertionError("expected NotImplementedError")

## The same class by hand and with `dataclass`

Most classes in pipelines only hold data. Writing `__init__`, `__repr__` and `__eq__` every time is repetitive. The
`@dataclass` decorator writes them for you from the **field annotations**: lines such as `name: str`, which list the
fields and their types.

Below, both versions are written out and asserted to behave the same way.

### Where you'll use it

- **Typed records:** an `Event` or `Order` dataclass as the in-memory shape of a row before it is written.
- **Serialisation:** `dataclasses.asdict` turns a record into a dictionary for JSON or a database insert.

In [ ]:
from dataclasses import dataclass

class OrderByHand:
    def __init__(self, order_id, amount):
        self.order_id = order_id
        self.amount = amount

    def __repr__(self):
        return f"OrderByHand(order_id={self.order_id!r}, amount={self.amount!r})"

    def __eq__(self, other):
        return (self.order_id, self.amount) == (other.order_id, other.amount)

@dataclass
class Order:
    order_id: int
    amount: float

assert OrderByHand(1, 9.5) == OrderByHand(1, 9.5)
assert Order(1, 9.5) == Order(1, 9.5)          # same behaviour, far less code
assert repr(Order(1, 9.5)) == "Order(order_id=1, amount=9.5)"

## Dataclass options

- `frozen=True` makes the object **read-only**: changing a field raises an error. Use it for values that must not change,
  such as a key or a configuration. A frozen object can also be a dictionary key.
- `field(default_factory=list)` gives each object its own new list. Writing `tags: list = []` would share one list between
  all objects, which is the mutable-default trap from chapter 3.
- `replace(obj, field=new)` returns a changed copy, so the original stays as it was.

### Where you'll use it

- **Immutable keys:** a frozen dataclass as a dictionary key for deduplication or caching.
- **Configuration:** a frozen dataclass that holds validated settings for a job, so no step can change them mid-run.

In [ ]:
from dataclasses import dataclass, field, replace

@dataclass(frozen=True)
class OrderKey:
    customer_id: int
    order_date: str

@dataclass
class Run:
    job: str
    attempts: int = 0                                 # a default value
    tags: list[str] = field(default_factory=list)     # a new list for every Run

first = Run("load")
second = Run("load")
first.tags.append("nightly")
assert second.tags == []                              # not shared

key = OrderKey(7, "2024-01-15")
assert key == OrderKey(7, "2024-01-15")
seen = {key: "processed"}                             # frozen objects can be dictionary keys
assert seen[OrderKey(7, "2024-01-15")] == "processed"

updated = replace(first, attempts=1)                  # a changed copy
assert updated.attempts == 1 and first.attempts == 0

try:
    key.customer_id = 8
except Exception:              # dataclasses.FrozenInstanceError is a subclass of AttributeError
    pass
else:
    raise AssertionError("frozen dataclasses cannot be changed")

## Enums: a fixed set of named values

An **enum** (enumeration) is a class whose objects are a fixed list of named values, such as the statuses a job can have.
Each member has a `name` (the label in code) and a `value` (the stored data). Comparing with a member catches typos that
a plain string would hide.

Create one by inheriting from `Enum`. Inheriting also from `str` lets a member compare equal to its raw text, which is
handy when the value comes from a file.

### Where you'll use it

- **Status fields:** `OK`, `FAILED`, `SKIPPED` for a job run, so a typo in a status string cannot reach a dashboard.
- **Source systems:** a fixed set of supported formats or regions, checked when a file arrives.

### Where you'll use it

- **Status fields:** `OK`, `FAILED`, `SKIPPED` for a job run, so a typo in a status string cannot reach a dashboard.
- **Source systems:** a fixed set of supported formats or regions, checked when a file arrives.

In [ ]:
from enum import Enum

class Status(str, Enum):
    OK = "ok"
    FAILED = "failed"
    SKIPPED = "skipped"

assert Status.OK.name == "OK"
assert Status.OK.value == "ok"
assert Status("failed") is Status.FAILED     # look up a member by its value
assert [s.value for s in Status] == ["ok", "failed", "skipped"]   # iterate over every member

def is_done(status):
    return status == Status.OK

assert is_done("ok")                          # str mixin: equal to the raw text
assert not is_done("failed")

try:
    Status("ok ")                             # the trailing space makes a different value
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

## Practice questions

1. Write a class `Counter` with an `increment()` method and a `value` property that starts at 0. Create two counters and
   show that they are independent.
2. Write a frozen dataclass `Money` with `amount` (a `Decimal`) and `currency` (a string), and a method `add(other)` that
   refuses to add two different currencies.
3. Write an `Enum` called `Region` with `NORTH`, `SOUTH` and `WEST`, and a function that returns the number of members.

## Answers

### Question 1

In [ ]:
class Counter:
    def __init__(self):
        self._value = 0

    def increment(self):
        self._value += 1

    @property
    def value(self):
        return self._value

a = Counter()
b = Counter()
a.increment()
a.increment()
assert a.value == 2 and b.value == 0

### Question 2

In [ ]:
from dataclasses import dataclass
from decimal import Decimal

@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str

    def add(self, other):
        if self.currency != other.currency:
            raise ValueError(f"cannot add {other.currency} to {self.currency}")
        return Money(self.amount + other.amount, self.currency)

assert Money(Decimal("10.00"), "EUR").add(Money(Decimal("2.50"), "EUR")) == Money(Decimal("12.50"), "EUR")
try:
    Money(Decimal("1"), "EUR").add(Money(Decimal("1"), "USD"))
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

### Question 3

In [ ]:
from enum import Enum

class Region(Enum):
    NORTH = "north"
    SOUTH = "south"
    WEST = "west"

assert len(Region) == 3
assert Region.WEST.value == "west"

## Interview notes

- **A class is a blueprint, an object is an instance.** `__init__` sets up each instance, and methods take `self`.
- **Use a dataclass for records and a plain class for behaviour.** A dataclass holds data; a class with many methods and
  hidden state is a service.
- **Never use a mutable default in a dataclass field.** Use `field(default_factory=list)`.
- **Frozen dataclasses give value semantics.** Equal fields mean equal objects, which makes them safe as dictionary keys.
- **Prefer composition to deep inheritance.** Two levels of subclassing are usually enough.